In [5]:
import pandas as pd
from bs4 import BeautifulSoup
import time
import random
import cloudscraper
import re
import json
import html

print("🚀 Starting the Fault-Tolerant Pure HTML TTRPG Scraper...")

# --- SETUP ---
# Set this to however many pages you want to scrape (100 pages = ~10,000 games)
MAX_PAGES_TO_SCRAPE = 1000
OUTPUT_FILENAME = "ttrpg_database_final.csv"

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

# ==========================================
# PHASE 1: HARVEST GAME LINKS (CORRECTED SESSION)
# ==========================================
import time
import random
from bs4 import BeautifulSoup
import re

game_links = []
MAX_PAGES_TO_SCRAPE = 100 

print(f"\n[PHASE 1] Harvesting {MAX_PAGES_TO_SCRAPE * 100} Game Links...")

# 🌟 THE FIX: We use the 'scraper' object directly. 
# It already acts as a persistent session and keeps your cookies!

for page in range(1, MAX_PAGES_TO_SCRAPE + 1):
    browse_url = f"https://boardgamegeek.com/browse/rpgitem/page/{page}"
    
    # Add a Referer to look like a real browsing path
    if page == 1:
        headers = {"Referer": "https://boardgamegeek.com/"}
    else:
        headers = {"Referer": f"https://boardgamegeek.com/browse/rpgitem/page/{page-1}"}

    print(f"  -> Scraping Browse Page {page:<3}...", end=" ")
    
    try:
        # Use scraper.get directly
        res = scraper.get(browse_url, headers=headers)
        
        # JITTER: Stay between 5 and 9 seconds to be very safe
        time.sleep(random.uniform(5.0, 9.0)) 
        
        if res.status_code == 200:
            # CHECK: Did Cloudflare give us a 'Just a moment' page?
            if "challenge-platform" in res.text or "Just a moment..." in res.text:
                print("| 🚨 Cloudflare Challenge detected! Try manually clearing in browser.")
                break

            soup = BeautifulSoup(res.content, 'html.parser')
            links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
            
            if not links:
                print("| 🚨 Zero links found. The page might be blank.")
                break

            for link in links:
                game_links.append(link['href'])
                
            current_unique = len(set(game_links))
            print(f"| 🎯 Total Unique Ready: {current_unique}")
            
            # --- THE COOLDOWN ---
            if page % 5 == 0:
                cooldown = random.randint(20, 40)
                print(f"     ☕ Cooldown... pausing for {cooldown}s...")
                time.sleep(cooldown)
                
        elif res.status_code == 403:
            print(f"| 🚨 403 Forbidden on Page {page}. IP is flagged.")
            break 
        else:
            print(f"| ⚠️ Failed. Status: {res.status_code}")
            
    except Exception as e:
        print(f"| Error: {e}")

game_links = list(set(game_links))
print(f"\n✅ Phase 1 Complete! Passing {len(game_links)} unique games to Phase 2.")

# ==========================================
# PHASE 2: SCRAPE INDIVIDUAL HTML PAGES
# ==========================================
print("\n[PHASE 2] Scraping Game Details (with Auto-Saving)...")
final_dataset = []

for index, link in enumerate(game_links):
    full_url = "https://boardgamegeek.com" + link
    search_name = link.split('/')[-1].replace('-', ' ').title() 
    
    # Print on the same line to keep the terminal clean
    print(f"  -> {index + 1}/{len(game_links)}: {search_name[:20]:<20}...", end=" ")
    
    try:
        res = scraper.get(full_url)
        # Jitter: Random sleep between 2.5 and 4.0 seconds (CRITICAL FOR HTML)
        time.sleep(random.uniform(2.5, 4.0)) 
        
        if res.status_code == 200:
            soup = BeautifulSoup(res.content, 'html.parser')
            
            # --- Extract Description ---
            desc = "No description"
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc:
                desc = meta_desc['content']
                desc = html.unescape(desc)
                desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                desc = re.sub(r'\n+', ' ', desc)
                desc = re.sub(r'\s{2,}', ' ', desc).strip()
                
            # --- Extract Stats & Real Name ---
            avg_score = 0.0
            num_reviews = 0
            
            scripts = soup.find_all('script')
            for script in scripts:
                if script.string and 'GEEK.geekitemPreload' in script.string:
                    json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                    if json_match:
                        try:
                            data = json.loads(json_match.group(1))
                            stats = data.get('item', {}).get('stats', {})
                            avg_score = float(stats.get('average', 0.0))
                            num_reviews = int(stats.get('usersrated', 0))
                            
                            official_name = data.get('item', {}).get('name')
                            if official_name:
                                search_name = official_name
                        except json.JSONDecodeError:
                            pass
                    break 
                    
            final_dataset.append({
                'Name': search_name,
                'Description': desc,
                'Average Score': avg_score,
                'Number of Reviews': num_reviews
            })
            
            print(f"| Score: {avg_score:.2f} | Reviews: {num_reviews}")
            
        elif res.status_code == 429:
            print("| 🚨 Rate Limited! Pausing for 30 seconds...")
            time.sleep(30)
        elif res.status_code == 403:
            print("| 🚨 403 Forbidden! Cloudflare blocked you. Wait 15 mins.")
            time.sleep(30)
        else:
            print(f"| ⚠️ Failed Status: {res.status_code}")
            
    except Exception as e:
        print(f"| Error: {e}")
        
    # --- AUTO-CHECKPOINT SYSTEM ---
    # Every 50 games, save the current progress to the CSV
    if (index + 1) % 50 == 0:
        pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
        print(f"     💾 Checkpoint Reached: Saved {index + 1} rows to {OUTPUT_FILENAME}")

# ==========================================
# FINAL SAVE
# ==========================================
df = pd.DataFrame(final_dataset)
df.to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 FULLY COMPLETE! Scraped {len(df)} games and saved to '{OUTPUT_FILENAME}'")

🚀 Starting the Fault-Tolerant Pure HTML TTRPG Scraper...

[PHASE 1] Harvesting 10000 Game Links...
  -> Scraping Browse Page 1  ... | 🎯 Total Unique Ready: 100
  -> Scraping Browse Page 2  ... | 🎯 Total Unique Ready: 200
  -> Scraping Browse Page 3  ... | 🎯 Total Unique Ready: 300
  -> Scraping Browse Page 4  ... | 🎯 Total Unique Ready: 400
  -> Scraping Browse Page 5  ... | 🎯 Total Unique Ready: 499
     ☕ Cooldown... pausing for 34s...
  -> Scraping Browse Page 6  ... | 🎯 Total Unique Ready: 600
  -> Scraping Browse Page 7  ... | 🎯 Total Unique Ready: 700
  -> Scraping Browse Page 8  ... | 🎯 Total Unique Ready: 800
  -> Scraping Browse Page 9  ... | 🎯 Total Unique Ready: 900
  -> Scraping Browse Page 10 ... | 🎯 Total Unique Ready: 1000
     ☕ Cooldown... pausing for 27s...
  -> Scraping Browse Page 11 ... | 🚨 Zero links found. The page might be blank.

✅ Phase 1 Complete! Passing 1000 unique games to Phase 2.

[PHASE 2] Scraping Game Details (with Auto-Saving)...
  -> 1/1000: The Str

In [6]:
df.head()

,Name,Description,Average Score,Number of Reviews
0,The Strange,Publisher Blurb: Beneath the orbits and atoms ...,7.86486,37
1,InSpectres,Fighting the Forces of Darkness so you don't h...,7.57355,138
2,Dungeons & Dragons Expert Set,The first version of the D&D Expert Set. It co...,7.97881,151
3,The Character Compendium,An unofficial supplement for Warhammer Fantasy...,8.09375,16
4,The Elves of Alfheim,"""This is the first Gazetteer to outline a non-...",7.24878,41


In [6]:
import os
import time
import random
import re
import json
import html
import math
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

print("🚀 Starting the Year-by-Year Interleaved TTRPG Scraper (10-Scrape Auto-Save)...")

# --- SETUP ---
OUTPUT_FILENAME = "ttrpg_database_final.csv"
START_YEAR = 2025  # Updated to start at 1985
END_YEAR = 2026
TARGET_TOTAL_GAMES = 10000

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://boardgamegeek.com/advsearch/rpgitem",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# ==========================================
# PRE-FLIGHT: LOAD EXISTING DATA
# ==========================================
existing_names = set()
normalized_existing_names = set() 
final_dataset = []
seen_hrefs = set() # Global tracker to prevent cross-year duplicate URLs
session_scrapes_count = 0 # Tracks successful Phase 2 scrapes for our 10-save trigger

if os.path.exists(OUTPUT_FILENAME):
    try:
        existing_df = pd.read_csv(OUTPUT_FILENAME)
        existing_names = set(existing_df['Name'].str.lower().dropna())
        
        normalized_existing_names = set(
            re.sub(r'[^a-z0-9]', '', str(name).lower()) for name in existing_names
        )
        
        final_dataset = existing_df.to_dict('records') 
        print(f"📦 Loaded {len(final_dataset)} existing games from CSV.")
    except Exception as e:
        print(f"⚠️ Could not load existing dataset: {e}")
else:
    print("⚠️ No existing CSV found. Starting fresh.")


# ==========================================
# MAIN LOOP: YEAR-BY-YEAR INTERLEAVED SCRAPING
# ==========================================
years_list = list(range(START_YEAR, END_YEAR + 1))

for i, year in enumerate(years_list):
    
    # 1. Calculate precise deficit based on actual saved games
    target_needed = TARGET_TOTAL_GAMES - len(final_dataset)
    
    if target_needed <= 0:
        print(f"\n🏁 TARGET REACHED! We have exactly {TARGET_TOTAL_GAMES} games in the dataset.")
        break

    remaining_years = len(years_list) - i
    current_year_quota = math.ceil((target_needed / remaining_years) * 1.10) 
    
    print(f"\n=======================================================")
    print(f"📅 STARTING YEAR: {year} | Target needed to hit 10k: {target_needed}")
    print(f"=======================================================")
    
    # ---------------------------------------------------------
    # PHASE 1: HARVEST LINKS FOR THIS YEAR ONLY
    # ---------------------------------------------------------
    year_links = [] 
    year_links_collected = 0
    print(f"  [PHASE 1] Harvesting Max {current_year_quota} Links (Pre-Skipping Duplicates)...")
    
    for page in range(1, 11): 
        if year_links_collected >= current_year_quota:
            print(f"    -> 🎯 Reached {current_year_quota} limit for {year}.")
            break

        if page == 1:
            search_url = f"https://boardgamegeek.com/search/rpgitem?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        else:
            search_url = f"https://boardgamegeek.com/search/rpgitem/page/{page}?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        
        print(f"    -> Scraping Page {page:<2}...", end=" ")
        
        try:
            res = scraper.get(search_url, headers=human_headers)
            time.sleep(random.uniform(4.0, 7.0)) 
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.content, 'html.parser')
                raw_links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
                
                if not raw_links:
                    print("| 🚨 End of pages for this year.")
                    break 
                
                previous_unique_count = len(seen_hrefs)
                unique_page_links = set([link['href'] for link in raw_links])
                
                added_this_page = 0
                skipped_this_page = 0
                
                for href in unique_page_links:
                    if year_links_collected >= current_year_quota:
                        break

                    if href not in seen_hrefs:
                        url_rough_name = re.sub(r'[^a-z0-9]', '', href.split('/')[-1].lower())
                        
                        if url_rough_name in normalized_existing_names:
                            seen_hrefs.add(href)
                            skipped_this_page += 1
                            continue 
                        
                        seen_hrefs.add(href)
                        year_links.append((href, year)) 
                        year_links_collected += 1
                        added_this_page += 1
                
                print(f"| Added: {added_this_page:<3} | Pre-Skipped: {skipped_this_page:<3} | 🎯 Year Total: {year_links_collected}/{current_year_quota}")
                
                if len(seen_hrefs) == previous_unique_count:
                    print("       ⏭️ BGG is repeating pages. Breaking pagination loop.")
                    break 
                
            elif res.status_code == 403:
                print("| 🚨 403 Blocked! Cloudflare caught us.")
                break 
            else:
                print(f"| ⚠️ Failed. Status: {res.status_code}")
                break
                
        except Exception as e:
            print(f"| Error: {e}")

    # ---------------------------------------------------------
    # PHASE 2: SCRAPE DETAILS FOR THIS YEAR ONLY
    # ---------------------------------------------------------
    if not year_links:
        print(f"  [PHASE 2] No new links found for {year}. Skipping to next year.")
        continue

    print(f"\n  [PHASE 2] Scraping Details for {len(year_links)} new games from {year}...")
    
    for index, (link, game_year) in enumerate(year_links):
        if len(final_dataset) >= TARGET_TOTAL_GAMES:
            print(f"\n🏁 TARGET REACHED DURING PHASE 2! Halting.")
            break

        full_url = "https://boardgamegeek.com" + link
        search_name = link.split('/')[-1].replace('-', ' ').title() 
        
        print(f"    -> {index + 1}/{len(year_links)}: [{game_year}] {search_name[:15]:<15}...", end=" ")
        
        if search_name.lower() in existing_names:
            print("| ⏭️ Skipped (Already in CSV - Caught in Phase 2)")
            continue
        
        try:
            res = scraper.get(full_url)
            time.sleep(random.uniform(2.5, 4.0)) 
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.content, 'html.parser')
                
                desc = "No description"
                meta_desc = soup.find('meta', attrs={'name': 'description'})
                if meta_desc:
                    desc = meta_desc['content']
                    desc = html.unescape(desc)
                    desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                    desc = re.sub(r'\n+', ' ', desc)
                    desc = re.sub(r'\s{2,}', ' ', desc).strip()
                    
                avg_score = 0.0
                num_reviews = 0
                
                scripts = soup.find_all('script')
                for script in scripts:
                    if script.string and 'GEEK.geekitemPreload' in script.string:
                        json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                        if json_match:
                            try:
                                data = json.loads(json_match.group(1))
                                stats = data.get('item', {}).get('stats', {})
                                avg_score = float(stats.get('average', 0.0))
                                num_reviews = int(stats.get('usersrated', 0))
                                
                                official_name = data.get('item', {}).get('name')
                                if official_name:
                                    search_name = official_name
                            except json.JSONDecodeError:
                                pass
                        break 
                        
                final_dataset.append({
                    'Name': search_name,
                    'Publishing Year': game_year,
                    'Description': desc,
                    'Average Score': avg_score,
                    'Number of Reviews': num_reviews
                })
                
                # Immediately update both tracking sets so Phase 1 knows about it next year
                existing_names.add(search_name.lower())
                normalized_existing_names.add(re.sub(r'[^a-z0-9]', '', search_name.lower()))
                
                print(f"| ✅ Score: {avg_score:.2f} | Total DB: {len(final_dataset)}")
                
                # 🌟 THE 10-SCRAPE AUTO-SAVE 🌟
                session_scrapes_count += 1
                if session_scrapes_count % 10 == 0:
                    pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
                    print(f"       💾 Auto-Save: 10 new games acquired. CSV safely overwritten.")
                
            elif res.status_code == 429:
                print("| 🚨 Rate Limited! Pausing for 30 seconds...")
                time.sleep(30)
            elif res.status_code == 403:
                print("| 🚨 403 Forbidden! Cloudflare blocked you.")
                time.sleep(30)
            else:
                print(f"| ⚠️ Failed Status: {res.status_code}")
                
        except Exception as e:
            print(f"| Error: {e}")

    # --- END OF YEAR CHECKPOINT ---
    pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
    
    cooldown = random.randint(15, 30)
    print(f"\n  💾 Checkpoint: {year} Complete & Saved. Total DB: {len(final_dataset)}")
    print(f"  ☕ Inter-year cooldown for {cooldown}s...")
    time.sleep(cooldown)

# ==========================================
# FINAL WRAP UP
# ==========================================
# One final save just in case the loop breaks before a multiple of 10 or end of year
pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 SCRIPT FINISHED! Database now has {len(final_dataset)} total games in '{OUTPUT_FILENAME}'")

🚀 Starting the Year-by-Year Interleaved TTRPG Scraper (10-Scrape Auto-Save)...
📦 Loaded 9911 existing games from CSV.

📅 STARTING YEAR: 2025 | Target needed to hit 10k: 89
  [PHASE 1] Harvesting Max 49 Links (Pre-Skipping Duplicates)...
    -> Scraping Page 1 ... | Added: 25  | Pre-Skipped: 77  | 🎯 Year Total: 25/49
    -> Scraping Page 2 ... | Added: 24  | Pre-Skipped: 0   | 🎯 Year Total: 49/49
    -> 🎯 Reached 49 limit for 2025.

  [PHASE 2] Scraping Details for 49 new games from 2025...
    -> 1/49: [2025] Monty Pythons C... | ✅ Score: 8.25 | Total DB: 9912
    -> 2/49: [2025] Hexploration De... | ✅ Score: 0.00 | Total DB: 9913
    -> 3/49: [2025] Terry Pratchett... | ✅ Score: 0.00 | Total DB: 9914
    -> 4/49: [2025] The World Below... | ✅ Score: 9.00 | Total DB: 9915
    -> 5/49: [2025] Conan The Hybor... | ✅ Score: 0.00 | Total DB: 9916
    -> 6/49: [2025] Teenage Mutant ... | ✅ Score: 7.65 | Total DB: 9917
    -> 7/49: [2025] Marvel Multiver... | ✅ Score: 0.00 | Total DB: 9918
 

In [4]:
import os
import time
import random
import re
import json
import html
import math
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

print("🚀 Starting the Autonomous TTRPG Scraper (3-Strike Rule & Auto-Restart)...")

# --- SETUP ---
OUTPUT_FILENAME = "ttrpg_database_final.csv"
START_YEAR = 2005  # Starting at 1989
END_YEAR = 2026
TARGET_TOTAL_GAMES = 10000

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://boardgamegeek.com/advsearch/rpgitem",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# ==========================================
# PRE-FLIGHT: LOAD EXISTING DATA
# ==========================================
existing_names = set()
normalized_existing_names = set() 
final_dataset = []
seen_hrefs = set() 
session_scrapes_count = 0 

if os.path.exists(OUTPUT_FILENAME):
    try:
        existing_df = pd.read_csv(OUTPUT_FILENAME)
        existing_names = set(existing_df['Name'].str.lower().dropna())
        
        normalized_existing_names = set(
            re.sub(r'[^a-z0-9]', '', str(name).lower()) for name in existing_names
        )
        
        final_dataset = existing_df.to_dict('records') 
        print(f"📦 Loaded {len(final_dataset)} existing games from CSV.")
    except Exception as e:
        print(f"⚠️ Could not load existing dataset: {e}")
else:
    print("⚠️ No existing CSV found. Starting fresh.")


# ==========================================
# MAIN LOOP: AUTONOMOUS INTERLEAVED SCRAPING
# ==========================================
years_list = list(range(START_YEAR, END_YEAR + 1))
consecutive_403_errors = 0
hard_exit = False

for i, year in enumerate(years_list):
    if hard_exit:
        break
        
    # THE RETRY LOOP: Keeps us on the same year if a 403 happens
    while True:
        target_needed = TARGET_TOTAL_GAMES - len(final_dataset)
        
        if target_needed <= 0:
            print(f"\n🏁 TARGET REACHED! We have exactly {TARGET_TOTAL_GAMES} games in the dataset.")
            hard_exit = True
            break

        remaining_years = len(years_list) - i
        current_year_quota = math.ceil((target_needed / remaining_years) * 1.10) 
        
        print(f"\n=======================================================")
        print(f"📅 STARTING YEAR: {year} | Target needed to hit 10k: {target_needed}")
        print(f"=======================================================")
        
        year_links = [] 
        year_links_collected = 0
        year_403_triggered = False
        
        # 🌟 Snapshot of seen links so we can cleanly revert if Phase 2 crashes
        seen_hrefs_snapshot = seen_hrefs.copy()
        
        print(f"  [PHASE 1] Harvesting Max {current_year_quota} Links (Pre-Skipping Duplicates)...")
        
        for page in range(1, 11): 
            if year_links_collected >= current_year_quota:
                print(f"    -> 🎯 Reached {current_year_quota} limit for {year}.")
                break

            if page == 1:
                search_url = f"https://boardgamegeek.com/search/rpgitem?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
            else:
                search_url = f"https://boardgamegeek.com/search/rpgitem/page/{page}?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
            
            print(f"    -> Scraping Page {page:<2}...", end=" ")
            
            try:
                res = scraper.get(search_url, headers=human_headers)
                time.sleep(random.uniform(4.0, 7.0)) # Restored original randomized timer
                
                if res.status_code == 200:
                    consecutive_403_errors = 0 # Reset strikes on success
                    soup = BeautifulSoup(res.content, 'html.parser')
                    raw_links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
                    
                    if not raw_links:
                        print("| 🚨 End of pages for this year.")
                        break 
                    
                    previous_unique_count = len(seen_hrefs)
                    unique_page_links = set([link['href'] for link in raw_links])
                    
                    added_this_page = 0
                    skipped_this_page = 0
                    
                    for href in unique_page_links:
                        if year_links_collected >= current_year_quota:
                            break

                        if href not in seen_hrefs:
                            url_rough_name = re.sub(r'[^a-z0-9]', '', href.split('/')[-1].lower())
                            
                            if url_rough_name in normalized_existing_names:
                                seen_hrefs.add(href)
                                skipped_this_page += 1
                                continue 
                            
                            seen_hrefs.add(href)
                            year_links.append((href, year)) 
                            year_links_collected += 1
                            added_this_page += 1
                    
                    print(f"| Added: {added_this_page:<3} | Pre-Skipped: {skipped_this_page:<3} | 🎯 Year Total: {year_links_collected}/{current_year_quota}")
                    
                    if len(seen_hrefs) == previous_unique_count:
                        print("       ⏭️ BGG is repeating pages. Breaking pagination loop.")
                        break 
                    
                elif res.status_code == 403:
                    consecutive_403_errors += 1
                    print(f"| 🚨 403 Blocked! (Strike {consecutive_403_errors}/3)")
                    year_403_triggered = True
                    break 
                else:
                    print(f"| ⚠️ Failed. Status: {res.status_code}")
                    break
                    
            except Exception as e:
                print(f"| Error: {e}")

        # --- Evaluate Phase 1 403 Trigger ---
        if consecutive_403_errors >= 3:
            print("\n🚨 3 CONSECUTIVE 403 ERRORS REACHED. INITIATING HARD SHUTDOWN. 🚨")
            hard_exit = True
            break
            
        if year_403_triggered:
            seen_hrefs = seen_hrefs_snapshot.copy() # Revert the link tracker
            print("     🔄 Cooling down for 60 seconds before restarting year from Phase 1...")
            time.sleep(60)
            continue # Restarts the while True loop

        # ---------------------------------------------------------
        # PHASE 2: SCRAPE DETAILS FOR THIS YEAR ONLY
        # ---------------------------------------------------------
        if not year_links:
            print(f"  [PHASE 2] No new links found for {year}. Skipping to next year.")
            break # Break while loop, go to next year

        print(f"\n  [PHASE 2] Scraping Details for {len(year_links)} new games from {year}...")
        
        for index, (link, game_year) in enumerate(year_links):
            if len(final_dataset) >= TARGET_TOTAL_GAMES:
                print(f"\n🏁 TARGET REACHED DURING PHASE 2! Halting.")
                hard_exit = True
                break

            full_url = "https://boardgamegeek.com" + link
            search_name = link.split('/')[-1].replace('-', ' ').title() 
            
            print(f"    -> {index + 1}/{len(year_links)}: [{game_year}] {search_name[:15]:<15}...", end=" ")
            
            if search_name.lower() in existing_names:
                print("| ⏭️ Skipped (Already in CSV - Caught in Phase 2)")
                continue
            
            try:
                res = scraper.get(full_url)
                time.sleep(random.uniform(2.5, 4.0)) # Restored original randomized timer
                
                if res.status_code == 200:
                    consecutive_403_errors = 0 # Reset strikes on success
                    soup = BeautifulSoup(res.content, 'html.parser')
                    
                    desc = "No description"
                    meta_desc = soup.find('meta', attrs={'name': 'description'})
                    if meta_desc:
                        desc = meta_desc['content']
                        desc = html.unescape(desc)
                        desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                        desc = re.sub(r'\n+', ' ', desc)
                        desc = re.sub(r'\s{2,}', ' ', desc).strip()
                        
                    avg_score = 0.0
                    num_reviews = 0
                    
                    scripts = soup.find_all('script')
                    for script in scripts:
                        if script.string and 'GEEK.geekitemPreload' in script.string:
                            json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                            if json_match:
                                try:
                                    data = json.loads(json_match.group(1))
                                    stats = data.get('item', {}).get('stats', {})
                                    avg_score = float(stats.get('average', 0.0))
                                    num_reviews = int(stats.get('usersrated', 0))
                                    
                                    official_name = data.get('item', {}).get('name')
                                    if official_name:
                                        search_name = official_name
                                except json.JSONDecodeError:
                                    pass
                            break 
                            
                    final_dataset.append({
                        'Name': search_name,
                        'Publishing Year': game_year,
                        'Description': desc,
                        'Average Score': avg_score,
                        'Number of Reviews': num_reviews
                    })
                    
                    existing_names.add(search_name.lower())
                    normalized_existing_names.add(re.sub(r'[^a-z0-9]', '', search_name.lower()))
                    
                    print(f"| ✅ Score: {avg_score:.2f} | Total DB: {len(final_dataset)}")
                    
                    session_scrapes_count += 1
                    if session_scrapes_count % 10 == 0:
                        pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
                        print(f"       💾 Auto-Save: 10 new games acquired. CSV safely overwritten.")
                    
                elif res.status_code == 429:
                    print("| 🚨 Rate Limited! Pausing for 30 seconds...")
                    time.sleep(30)
                elif res.status_code == 403:
                    consecutive_403_errors += 1
                    print(f"| 🚨 403 Blocked! (Strike {consecutive_403_errors}/3)")
                    year_403_triggered = True
                    break 
                else:
                    print(f"| ⚠️ Failed Status: {res.status_code}")
                    
            except Exception as e:
                print(f"| Error: {e}")

        # --- Evaluate Phase 2 403 Trigger ---
        if hard_exit:
            break
            
        if consecutive_403_errors >= 3:
            print("\n🚨 3 CONSECUTIVE 403 ERRORS REACHED. INITIATING HARD SHUTDOWN. 🚨")
            hard_exit = True
            break
            
        if year_403_triggered:
            seen_hrefs = seen_hrefs_snapshot.copy() # Revert the link tracker
            print("     🔄 Cooling down for 60 seconds before restarting year from Phase 1...")
            time.sleep(60)
            continue # Restarts the while True loop

        # --- END OF YEAR CHECKPOINT ---
        pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
        
        cooldown = random.randint(15, 30)
        print(f"\n  💾 Checkpoint: {year} Complete & Saved. Total DB: {len(final_dataset)}")
        print(f"  ☕ Inter-year cooldown for {cooldown}s...")
        time.sleep(cooldown)
        
        # Break the retry loop to successfully move to the next year
        break 

# ==========================================
# FINAL WRAP UP
# ==========================================
pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 SCRIPT SECURELY TERMINATED! Database safely contains {len(final_dataset)} total games in '{OUTPUT_FILENAME}'")

🚀 Starting the Autonomous TTRPG Scraper (3-Strike Rule & Auto-Restart)...
📦 Loaded 9901 existing games from CSV.

📅 STARTING YEAR: 2005 | Target needed to hit 10k: 99
  [PHASE 1] Harvesting Max 5 Links (Pre-Skipping Duplicates)...
    -> Scraping Page 1 ... | Added: 5   | Pre-Skipped: 17  | 🎯 Year Total: 5/5
    -> 🎯 Reached 5 limit for 2005.

  [PHASE 2] Scraping Details for 5 new games from 2005...
    -> 1/5: [2005] Borderlands And... 

KeyboardInterrupt: 

In [12]:
import pandas as pd

finalfinaldf = pd.read_csv("ttrpg_database_final.csv")

finalfinaldf["Publishing Year"].value_counts()
# finalfinaldf.head()

Publishing Year
1989.0    406
1992.0    377
2003.0    363
1999.0    349
2005.0    306
1990.0    271
2010.0    265
1996.0    247
1982.0    246
1988.0    240
2015.0    236
1983.0    211
1984.0    211
1986.0    210
1985.0    209
1987.0    209
1981.0    202
1991.0    198
1993.0    191
1995.0    190
1994.0    190
1998.0    186
1997.0    186
2000.0    177
2001.0    176
2002.0    176
2004.0    166
2006.0    157
2007.0    156
2008.0    155
2009.0    154
2021.0    146
2011.0    145
2012.0    143
2013.0    142
2014.0    141
1980.0    139
2025.0    132
2017.0    128
2016.0    128
2018.0    127
2019.0    125
2020.0    123
2022.0    113
2023.0    110
2024.0    106
2026.0     50
Name: count, dtype: int64

In [15]:
import os
import time
import random
import re
import json
import urllib.parse
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

print("🚀 Starting the HTML-Only Autonomous Year Patcher (Forced Int Edition)...")

# --- SETUP ---
OUTPUT_FILENAME = "ttrpg_database_final.csv"

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# ==========================================
# 1. CLEAN SLATE LOAD & TYPE CASTING
# ==========================================
try:
    df = pd.read_csv(OUTPUT_FILENAME)
    
    # 🌟 THE FIX: Force the entire column to Pandas' Nullable Integer type (Int64)
    # errors='coerce' turns string junk like "Unknown" or "N/A" into proper empty nulls (<NA>)
    df['Publishing Year'] = pd.to_numeric(df['Publishing Year'], errors='coerce').astype('Int64')
    
    print(f"📦 Loaded dataset with {len(df)} total games.")
except FileNotFoundError:
    print(f"🚨 Error: Could not find '{OUTPUT_FILENAME}'.")
    exit()

# Identify missing years (Now much simpler because of the numeric coercion above)
missing_mask = df['Publishing Year'].isna() | (df['Publishing Year'] == 0)

missing_df = df[missing_mask]
missing_indices = missing_df.index.tolist()

if not missing_indices:
    print("\n✅ Your dataset is perfectly clean! No missing years found.")
    exit()

print(f"\n🔍 Found {len(missing_indices)} games missing their Publishing Year.")
print("Initiating HTML search-and-scrape sequence...\n")

# ==========================================
# 2. HTML SEARCH & SCRAPE LOOP
# ==========================================
session_scrapes_count = 0
consecutive_403 = 0
hard_exit = False

for idx in missing_indices:
    if hard_exit:
        break
        
    game_name = str(df.at[idx, 'Name'])
    print(f"  -> Patching: {game_name[:25]:<25}...", end=" ")
    
    # STEP A: Search the name to find the exact BGG URL
    encoded_name = urllib.parse.quote(game_name)
    search_url = f"https://boardgamegeek.com/search/rpgitem?q={encoded_name}"
    
    try:
        res_search = scraper.get(search_url, headers=human_headers)
        time.sleep(random.uniform(2.5, 4.0)) 
        
        if res_search.status_code == 403:
            consecutive_403 += 1
            print(f"| 🚨 403 on Search! (Strike {consecutive_403}/3)")
            if consecutive_403 >= 3: hard_exit = True
            continue
            
        consecutive_403 = 0
        soup_search = BeautifulSoup(res_search.content, 'html.parser')
        first_link = soup_search.find('a', href=re.compile(r'^/rpgitem/\d+/'))
        
        if not first_link:
            print("| ⏭️ No search results found.")
            continue
            
        full_url = "https://boardgamegeek.com" + first_link['href']
        
        # STEP B: Scrape the actual game page
        res_page = scraper.get(full_url, headers=human_headers)
        time.sleep(random.uniform(2.5, 4.0)) 
        
        if res_page.status_code == 403:
            consecutive_403 += 1
            print(f"| 🚨 403 on Game Page! (Strike {consecutive_403}/3)")
            if consecutive_403 >= 3: hard_exit = True
            continue
            
        consecutive_403 = 0
        soup_page = BeautifulSoup(res_page.content, 'html.parser')
        scripts = soup_page.find_all('script')
        
        found_year = False
        for script in scripts:
            if script.string and 'GEEK.geekitemPreload' in script.string:
                json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                if json_match:
                    try:
                        data = json.loads(json_match.group(1))
                        year = data.get('item', {}).get('yearpublished')
                        
                        if year and str(year).strip() not in ['0', '']:
                            # 🌟 THE FIX: Explicitly cast the scraped value to int before saving
                            df.at[idx, 'Publishing Year'] = int(year)
                            found_year = True
                            print(f"| ✅ Year: {int(year)}")
                            
                    except (json.JSONDecodeError, ValueError):
                        pass
                break
                
        if not found_year:
            print("| ⚠️ Year missing on page.")
            
        # 10-Scrape Auto-Save
        session_scrapes_count += 1
        if session_scrapes_count % 10 == 0:
            df.to_csv(OUTPUT_FILENAME, index=False)
            print(f"       💾 Auto-Save: Cleaned 10 rows. CSV safely overwritten.")
            
    except Exception as e:
        print(f"| Error: {e}")

# ==========================================
# FINAL WRAP UP
# ==========================================
df.to_csv(OUTPUT_FILENAME, index=False)

if hard_exit:
    print(f"\n🚨 SCRIPT TERMINATED DUE TO 403 STRIKES! Progress safely saved to '{OUTPUT_FILENAME}'.")
else:
    print(f"\n🎉 HTML Patching complete! Clean dataset saved to '{OUTPUT_FILENAME}'.")

🚀 Starting the HTML-Only Autonomous Year Patcher (Forced Int Edition)...
📦 Loaded dataset with 10000 total games.

🔍 Found 986 games missing their Publishing Year.
Initiating HTML search-and-scrape sequence...

  -> Patching: The Strange              ... | ✅ Year: 2014
  -> Patching: InSpectres               ... | ✅ Year: 2004
  -> Patching: Dungeons & Dragons Expert... | ✅ Year: 1981
  -> Patching: The Character Compendium ... | ✅ Year: 2013
  -> Patching: The Elves of Alfheim     ... | ✅ Year: 1988
  -> Patching: The Inner Sea World Guide... | ✅ Year: 2011
  -> Patching: Fudge                    ... | ✅ Year: 1995
  -> Patching: Wizard's Spell Compendium... | ✅ Year: 1996
  -> Patching: The Shamutanti Hills     ... | ✅ Year: 2009
  -> Patching: The Pendragon Campaign   ... | ✅ Year: 1985
       💾 Auto-Save: Cleaned 10 rows. CSV safely overwritten.
  -> Patching: Swashbucklers of the 7 Sk... | ✅ Year: 2009
  -> Patching: Ninth World Guidebook    ... | ✅ Year: 2015
  -> Patching: Dunge